# PSO experiments on Kaggle (resumable)

**Settings (right panel):** Accelerator **GPU T4 x2** · Internet **On** · Add-ons → Secrets → `HF_TOKEN` (your Hugging Face token, attached to this notebook).

**Inputs:** add the dataset that holds `Final_PSO.zip` (the code). From the 2nd run on, also add the **output of the previous version of this notebook** (Add Input → Your Work → this notebook) — its `outputs/` folder is restored automatically and every experiment resumes from its cache.

**How to run:** set `STAGE` below, then **Save Version → Save & Run All (Commit)**. The run stops itself before Kaggle's 12-hour limit, so the output is always saved. Re-run the same stage until it prints `Stage X complete`, then move to the next stage.

| Stage | What | Approx. T4×2 time |
|---|---|---|
| A | Llama-2: main corpus (5 seeds), Exp 1, 2, 3, 5, 6, 8, 9, 10 | 6–8 h |
| B | Llama-2: attacks (Exp 4) | 10–14 h (may need 2 runs) |
| C | Exp 7: Llama-2 benchmarks, Llama-3, Mistral | 8–12 h (may need 2 runs) |
| D | per-prompt logs + summary (fast) | minutes |

Before stage C, accept the licences on Hugging Face (same account as the token) for `meta-llama/Llama-2-7b-chat-hf`, `meta-llama/Meta-Llama-3-8B-Instruct` and `mistralai/Mistral-7B-Instruct-v0.2`.

In [ ]:
STAGE = "ALL"            # "ALL" runs A -> B -> C -> D, resuming where the last run stopped
FAST = True              # 3 seeds, smaller attack subsets
TIME_BUDGET_HOURS = 11.0  # stops cleanly before Kaggle's 12 h limit

In [ ]:
import glob, os, shutil, subprocess, sys
from pathlib import Path

# 1) Code: copy Final_PSO from the input dataset into the writable working directory.
REPO = Path("/kaggle/working/Final_PSO")
if not REPO.exists():
    src = sorted((p for p in glob.glob("/kaggle/input/**/Final_PSO", recursive=True) if Path(p, "experiments").is_dir()),
                 key=lambda p: Path(p, "outputs").exists())  # prefer the code dataset over old outputs
    zips = glob.glob("/kaggle/input/**/Final_PSO*.zip", recursive=True)
    if src:
        shutil.copytree(src[0], REPO, ignore=shutil.ignore_patterns("outputs"))
    elif zips:
        shutil.unpack_archive(zips[0], "/kaggle/working")
    else:
        raise FileNotFoundError("Add the dataset containing Final_PSO (or Final_PSO.zip) as an input.")
sys.path.insert(0, str(REPO / "kaggle"))
import kaggle_runner as kr
kr.restore_previous_outputs(REPO)

# 2) Environment
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.56", "accelerate>=0.26"], check=True)
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret attached; using the ungated, weight-identical model repositories in config.yaml.")
os.environ["HF_HOME"] = "/tmp/hf"               # model weights stay out of /kaggle/working (20 GB limit)
os.environ["PSO_DELETE_MODEL_CACHE"] = "1"       # delete a backbone's weights after Exp 7 is done with it
os.environ["TOKENIZERS_PARALLELISM"] = "false"
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])

# 3) Data (downloads public benchmarks, builds the constructed sets; skipped if present)
subprocess.run([sys.executable, "data/prepare_data.py"], cwd=REPO, check=True)
kr.status(REPO)

In [ ]:
if STAGE == "ALL":
    kr.run_all(REPO, TIME_BUDGET_HOURS, fast=FAST)
else:
    kr.run_stage(REPO, STAGE, TIME_BUDGET_HOURS, fast=FAST)
kr.status(REPO)

In [ ]:
# Small archive of results to download (tables, figures, thresholds, logs; not the large per-run caches).
out = REPO / "outputs"
keep = Path("/kaggle/working/pso_results")
shutil.rmtree(keep, ignore_errors=True)
for sub in ("tables", "figures", "logs"):
    if (out / sub).exists():
        shutil.copytree(out / sub, keep / sub)
for f in out.glob("*.json"):
    shutil.copy(f, keep)
for f in out.glob("runs/*/thresholds.json"):
    (keep / "thresholds").mkdir(parents=True, exist_ok=True)
    shutil.copy(f, keep / "thresholds" / f"{f.parent.name}.json")
shutil.make_archive("/kaggle/working/pso_results", "zip", keep)
print("Download /kaggle/working/pso_results.zip from the Output tab; the full outputs/ folder is kept for resuming.")